In [1]:
import pandas as pd
from fastparquet import write
from fastparquet import ParquetFile
from pathlib import Path
import numpy as np
import os

In [2]:
data_pipeline = "ohe"
input_pipeline = "baseline"

In [3]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "../../kaggle/input/datasets/abhinavneelam/smartphone-addiction/data/"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data/"
    output_path = "../../"

In [4]:
ss = pd.read_csv("../../data/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

'addicted_label'

In [5]:
cat_cols = ["gender"]
X = ParquetFile(Path(data_path) / f"{input_pipeline}/train.parq").to_pandas()
X_test = ParquetFile(Path(data_path) / f"{input_pipeline}/test.parq").to_pandas()

for ft in cat_cols:
    if ft not in X:
        continue

    X = pd.concat([X, pd.get_dummies(X[ft])], axis=1)
    X.drop(ft, axis=1, inplace=True)

    X_test = pd.concat([X_test, pd.get_dummies(X_test[ft])], axis=1)
    X_test.drop(ft, axis=1, inplace=True)

out_path = Path(data_path) / f"{data_pipeline}"
out_path.mkdir(parents=True, exist_ok=True)

write(out_path / f"train_{input_pipeline}.parq", X)
write(out_path / f"test_{input_pipeline}.parq", X_test)